In [127]:
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

## Load database

In [128]:
df_patients = pd.read_csv("../database/patients.csv")
df_services_weekly = pd.read_csv("../database/services_weekly.csv")
df_staff_schedule = pd.read_csv("../database/staff_schedule.csv")
df_staff = pd.read_csv("../database/staff.csv")

print(f"Shape of df_patients: {df_patients.shape}")
print(f"Shape of df_services_weekly: {df_services_weekly.shape}")
print(f"Shape of df_staff_schedule: {df_staff_schedule.shape}")
print(f"Shape of df_staff: {df_staff.shape}")

Shape of df_patients: (1000, 7)
Shape of df_services_weekly: (208, 10)
Shape of df_staff_schedule: (6552, 6)
Shape of df_staff: (110, 4)


In [129]:
print(df_patients.head(2))
print(df_services_weekly.head(2))
print(df_staff_schedule.head(2))
print(df_staff.head(2))

     patient_id               name  age arrival_date departure_date  service  \
0  PAT-09484753  Richard Rodriguez   24   2025-03-16     2025-03-22  surgery   
1  PAT-f0644084     Shannon Walker    6   2025-12-13     2025-12-14  surgery   

   satisfaction  
0            61  
1            83  
   week  month    service  available_beds  patients_request  \
0     1      1  emergency              32                76   
1     1      1    surgery              45               130   

   patients_admitted  patients_refused  patient_satisfaction  staff_morale  \
0                 32                44                    67            70   
1                 45                85                    83            78   

  event  
0  none  
1   flu  
   week      staff_id    staff_name    role    service  present
0     1  STF-b77cdc60  Allison Hill  doctor  emergency        1
1     2  STF-b77cdc60  Allison Hill  doctor  emergency        1
       staff_id    staff_name    role    service
0  STF-5c

## Merge database

In [130]:
df_patients['arrival_date'] = pd.to_datetime(df_patients['arrival_date'])
df_patients['departure_date'] = pd.to_datetime(df_patients['departure_date'])
df_patients['month'] = df_patients['arrival_date'].dt.month
df_patients['week'] = (df_patients['arrival_date'].dt.day - 1) // 7 + 1

# One record per: week + service + staff_id
df_staff_week = (
    df_staff_schedule
    .groupby(["week", "service", "staff_id"], as_index=False)
    .agg(present=("present", "max"))
)

# Aggregate staff schedule
df_staff_weekly = (
    df_staff_week
    .groupby(["week", "service"], as_index=False)
    .agg(
        total_staff=('staff_id', 'nunique'),
        staff_present=("present", "sum")
    )
    .reset_index()
)

# Add staff information
df_staff_weekly["staff_absens"] = df_staff_weekly["total_staff"] - df_staff_weekly["staff_present"]
df_staff_weekly["staff_check"] = df_staff_weekly["staff_present"] + df_staff_weekly["staff_absens"]

print("\nStaff consistency:")
print((df_staff_weekly["staff_check"]==df_staff_weekly["total_staff"]).all())

# Remove validaton column
df_staff_weekly = df_staff_weekly.drop(columns=["staff_check"])

# Compute service statistics
df_service = df_services_weekly.merge(df_staff_weekly, on=["week", "service"], how="left")

df = df_patients.merge(df_service, on=["week", "service"], how="left")


Staff consistency:
True


## Fill missing value

In [131]:
# Round the values  to 2 decimal places
numeric_cols = [
    "available_beds",
    "patients_request",
    "patients_admitted",
    "patients_refused",
    "patient_satisfaction",
    "staff_morale",
    "total_staff",
    "staff_present",
    "staff_absens"
]

# Make available beds column into integer type
df["available_beds"] = df["available_beds"].astype(int)
df['total_staff'] = df['total_staff'].astype(int)

df[numeric_cols] = df[numeric_cols].round(2)

# Drop the "event" column if it exists
df = df.drop(columns=["event"])

In [132]:
df.head(2)

,patient_id,name,age,arrival_date,departure_date,service,satisfaction,month_x,week,month_y,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,index,total_staff,staff_present,staff_absens
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61,3,3,1,27,66,27,39,63,72,11,25,0,25
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83,12,2,1,40,26,26,0,96,56,7,25,23,2


In [133]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 1000 entries, 0 to 999
Data columns (total 20 columns):
 #   Column                Non-Null Count  Dtype         
---  ------                --------------  -----         
 0   patient_id            1000 non-null   object        
 1   name                  1000 non-null   object        
 2   age                   1000 non-null   int64         
 3   arrival_date          1000 non-null   datetime64[ns]
 4   departure_date        1000 non-null   datetime64[ns]
 5   service               1000 non-null   object        
 6   satisfaction          1000 non-null   int64         
 7   month_x               1000 non-null   int32         
 8   week                  1000 non-null   int32         
 9   month_y               1000 non-null   int64         
 10  available_beds        1000 non-null   int64         
 11  patients_request      1000 non-null   int64         
 12  patients_admitted     1000 non-null   int64         
 13  patients_refused   

## Develop data into 10,000 rows

In [138]:
import numpy as np
from faker import Faker

# Default Configurations
DEFAULT_TARGET_ROWS = 10_000
DEFAULT_SEED = 42

COLUMN_ORDER = [
    "patient_id", "name", "age", "arrival_date", "departure_date",
    "service", "satisfaction", "month", "week", "available_beds",
    "patients_request", "patients_admitted", "patients_refused",
    "patient_satisfaction", "staff_morale", "total_staff",
    "staff_present", "staff_absens"
]

DECIMAL_COLUMNS = [
    "patients_request", "patients_admitted", "patients_refused",
    "patient_satisfaction", "staff_morale", "staff_present", "staff_absens"
]


def generate_synthetic_data(
    df: pd.DataFrame, 
    target_rows: int = DEFAULT_TARGET_ROWS, 
    seed: int = DEFAULT_SEED
) -> pd.DataFrame:
    """
    Generates synthetic patient records based on existing DataFrame feature distributions.
    """
    np.random.seed(seed)
    Faker.seed(seed)
    fake = Faker()

    current_rows = len(df)
    additional_rows = max(0, target_rows - current_rows)

    print(f"Current rows : {current_rows}")
    print(f"Rows to add  : {additional_rows}")

    if additional_rows == 0:
        return df[COLUMN_ORDER].copy()

    # ------------------------------------------------------------
    # 1. REFERENCE DISTRIBUTIONS & STATS
    # ------------------------------------------------------------
    service_dist = df["service"].value_counts(normalize=True)
    services, service_probs = service_dist.index.to_numpy(), service_dist.values

    hospital_metrics = [
        "available_beds", "patients_request", "patients_admitted",
        "patients_refused", "patient_satisfaction", "staff_morale",
        "total_staff", "staff_present", "staff_absens"
    ]
    
    # Calculate mean and std per service
    service_stats = (
        df.groupby("service")[hospital_metrics]
        .agg(["mean", "std"])
        .fillna(0)
    )

    # ------------------------------------------------------------
    # 2. VECTORIZED BASE FEATURE GENERATION
    # ------------------------------------------------------------
    chosen_services = np.random.choice(services, size=additional_rows, p=service_probs)
    ages = np.clip(np.random.normal(df["age"].mean(), df["age"].std(), additional_rows), 0, 100).astype(int)
    satisfaction = np.clip(np.random.normal(df["satisfaction"].mean(), df["satisfaction"].std(), additional_rows), 0, 100).astype(int)

    # Dates Generation (Vectorized)
    start_ts = pd.Timestamp("2025-01-01").value
    end_ts = pd.Timestamp("2025-12-31").value
    random_ts = np.random.randint(start_ts, end_ts, size=additional_rows, dtype="int64")
    
    arrival_dates = pd.to_datetime(random_ts).floor("D")
    length_of_stay = pd.to_timedelta(np.random.randint(1, 15, size=additional_rows), unit="D")
    departure_dates = arrival_dates + length_of_stay

    # Assemble Base Data Frame
    synthetic_df = pd.DataFrame({
        "patient_id": [f"PAT-{fake.hexify(text='^^^^^^^^')}" for _ in range(additional_rows)],
        "name": [fake.name() for _ in range(additional_rows)],
        "age": ages,
        "arrival_date": arrival_dates,
        "departure_date": departure_dates,
        "service": chosen_services,
        "satisfaction": satisfaction,
        "month": arrival_dates.month,
        "week": ((arrival_dates.day - 1) // 7) + 1,
    })

    # ------------------------------------------------------------
    # 3. SERVICE-BASED HOSPITAL FEATURES (VECTORIZED MAP)
    # ------------------------------------------------------------
    for col in hospital_metrics:
        means = synthetic_df["service"].map(service_stats[(col, "mean")])
        stds = synthetic_df["service"].map(service_stats[(col, "std")])
        samples = np.random.normal(means, stds)

        if col in ["available_beds", "total_staff"]:
            min_val = 0 if col == "available_beds" else 1
            synthetic_df[col] = np.maximum(min_val, np.round(samples)).astype(int)
        elif col in ["patient_satisfaction", "staff_morale"]:
            synthetic_df[col] = np.clip(samples, 0, 100)
        else:  # Count/request metrics
            synthetic_df[col] = np.maximum(0, samples)

    # ------------------------------------------------------------
    # 4. MERGE & FORMATTING
    # ------------------------------------------------------------
    df_combined = pd.concat([df, synthetic_df], ignore_index=True)[COLUMN_ORDER]
    df_combined[DECIMAL_COLUMNS] = df_combined[DECIMAL_COLUMNS].round(2)

    print(f"\nFinal dataset shape: {df_combined.shape}")
    return df_combined


# Example Usage:
df_10k = generate_synthetic_data(df, target_rows=10_000)

Current rows : 1000
Rows to add  : 9000

Final dataset shape: (10000, 18)


In [139]:
df_10k

,patient_id,name,age,arrival_date,departure_date,service,satisfaction,month,week,available_beds,patients_request,patients_admitted,patients_refused,patient_satisfaction,staff_morale,total_staff,staff_present,staff_absens
0,PAT-09484753,Richard Rodriguez,24,2025-03-16,2025-03-22,surgery,61,NaN,3,27,66.00,27.00,39.00,63.00,72.00,25,0.00,25.00
1,PAT-f0644084,Shannon Walker,6,2025-12-13,2025-12-14,surgery,83,NaN,2,40,26.00,26.00,0.00,96.00,56.00,25,23.00,2.00
2,PAT-ac6162e4,Julia Torres,24,2025-06-29,2025-07-05,general_medicine,83,NaN,5,40,103.00,40.00,63.00,73.00,52.00,28,23.00,5.00
3,PAT-3dda2bb5,Crystal Johnson,32,2025-10-12,2025-10-23,emergency,81,NaN,2,28,169.00,28.00,141.00,75.00,64.00,39,37.00,2.00
4,PAT-08591375,Garrett Lin,25,2025-02-18,2025-02-25,ICU,76,NaN,3,20,21.00,20.00,1.00,82.00,89.00,34,0.00,34.00
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
9995,PAT-bc982acf,Michael Sparks,73,2025-03-09,2025-03-10,surgery,84,3.0,2,53,69.56,23.03,41.28,79.74,97.14,25,14.31,7.64
9996,PAT-94ea6637,James Baker,24,2025-02-24,2025-03-08,emergency,81,2.0,4,29,122.29,30.06,109.46,65.76,73.29,39,48.14,16.79
9997,PAT-042af740,Ashley Gentry,78,2025-11-17,2025-11-22,ICU,83,11.0,3,19,20.71,10.77,4.51,73.55,87.58,34,40.45,0.00
9998,PAT-6d3c54f9,Laura Rodgers,48,2025-03-09,2025-03-11,emergency,76,3.0,2,32,215.30,30.21,178.38,64.36,59.46,39,18.64,0.00
